In [ ]:
import re
import warnings
import torch
import torch.nn.functional as F
from torch.optim import SGD
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from tqdm import tqdm

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
G = 4                # completions per prompt
MAX_NEW = 2048       # max new tokens (more headroom for math reasoning)
LR = 1e-3
EPOCHS = 1
CLIP_EPS = 0.2       # PPO clip epsilon
KL_COEF = 0.01       # KL penalty coefficient
TEMPERATURE = 0.9
TRAIN_SIZE = 50      # GSM8K training subset
EVAL_SIZE = 20       # GSM8K eval subset

print(f"Using device: {DEVICE}")

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
def extract_answer(text: str) -> str | None:
    """Extract the final numeric answer from a GSM8K solution string.
    GSM8K solutions end with '#### <number>'."""
    match = re.search(r"####\s*([\d,\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()
    return None

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful math tutor. Solve problems step by step. "
    "At the end, write your final answer after '####'."
)

def build_messages(question: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

def apply_chat(messages: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    """Tokenize a chat conversation and return (input_ids, attention_mask) tensors on DEVICE."""
    template = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
    )
    input_ids = template['input_ids']
    mask = template['attention_mask']

    return torch.tensor([input_ids], dtype=torch.long, device=DEVICE), torch.tensor([mask], dtype=torch.long, device=DEVICE)

In [ ]:
print("Loading GSM8K dataset...")
raw = load_dataset("openai/gsm8k", "main")

train_items = [
    {"messages": build_messages(ex["question"]), "answer": extract_answer(ex["answer"])}
    for ex in raw["train"].select(range(TRAIN_SIZE))
    if extract_answer(ex["answer"]) is not None
]
eval_items = [
    {"messages": build_messages(ex["question"]), "answer": extract_answer(ex["answer"])}
    for ex in raw["test"].select(range(EVAL_SIZE))
    if extract_answer(ex["answer"]) is not None
]

print(f"Train samples: {len(train_items)} | Eval samples: {len(eval_items)}")

In [ ]:
# LoRA config
LORA_R = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# ── Model ─────────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Policy model: base weights frozen, only LoRA adapters are trainable
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model = model.to(DEVICE)
model.train()
model.print_trainable_parameters()

# Frozen reference model for KL penalty
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)
ref_model = ref_model.to(DEVICE)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

In [ ]:
optimizer = SGD(model.parameters(), lr=LR)

# ── Reward function ────────────────────────────────────────────────────────────
def reward_fn(completion: str, gold_answer: str) -> float:
    """Multi-signal reward for GSM8K:
    - 1.0  exact match on the '#### <n>' final answer
    - 0.5  gold number appears anywhere in the completion
    - 0.1  completion contains any digit (partial credit)
    - 0.0  otherwise
    """
    pred = extract_answer(completion)
    if pred is not None and pred == gold_answer:
        return 1.0
    if re.search(rf"\b{re.escape(gold_answer)}\b", completion):
        return 0.5
    if re.search(r"\d", completion):
        return 0.1
    return 0.0

In [ ]:
# ── Eval ──────────────────────────────────────────────────────────────────────
@torch.inference_mode()
def evaluate(items: list[dict], tag: str = "eval") -> float:
    """Greedy decode one completion per item, report mean reward."""
    model.eval()
    total_reward = 0.0
    total = len(items)

    print(f"\n{'─'*60}")
    print(f"[{tag}] Evaluating on {total} items...")

    for i, item in tqdm(enumerate(items), total=total, desc=f"[{tag}] Evaluating"):
        input_ids, mask = apply_chat(item["messages"])
        prompt_len = input_ids.shape[1]

        out = model.generate(
            input_ids=input_ids,
            attention_mask=mask,
            max_new_tokens=MAX_NEW,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
        completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)
        gold = item["answer"]
        r = reward_fn(completion, gold)
        total_reward += r

        if i < 3:  # show a few examples
            print(f"  [r={r:.1f}] gold={gold!r} | {completion[:80]!r}")

    mean_reward = total_reward / total
    print(f"[{tag}] Mean reward: {mean_reward:.4f}")
    print(f"{'─'*60}\n")
    model.train()
    return mean_reward

# ── Helpers ───────────────────────────────────────────────────────────────────
@torch.inference_mode()
def sample_completions(messages: list[dict], n: int):
    """Sample n completions for a chat prompt. Returns (input_ids list, completion strings, prompt_len, mask)."""
    input_ids, mask = apply_chat(messages)
    prompt_len = input_ids.shape[1]

    all_ids = []
    all_texts = []

    for i in range(n):
        out = model.generate(
            input_ids=input_ids,
            attention_mask=mask,
            max_new_tokens=MAX_NEW,
            do_sample=True,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.eos_token_id,
        )
        all_ids.append(out)
        completion_ids = out[0, prompt_len:]
        completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True)
        all_texts.append(completion_text)

        print(f"  sample {i + 1}/{n}: {completion_text!r}")

    return all_ids, all_texts, prompt_len

# ── GRPO step ─────────────────────────────────────────────────────────────────
def grpo_step(messages: list[dict], gold_answer: str):
    # 1. Sample G completions
    all_ids, completions, prompt_len = sample_completions(messages, G)

    # 2. Score with reward function
    rewards = torch.tensor(
        [reward_fn(c, gold_answer) for c in completions],
        dtype=torch.bfloat16, device=DEVICE
    )

    # 3. Normalize rewards within group (GRPO advantage)
    if rewards.std() > 1e-8:
        advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    else:
        advantages = rewards - rewards.mean()

    total_loss = torch.tensor(0.0, dtype=torch.bfloat16, device=DEVICE, requires_grad=True)

    for i, (seq_ids, advantage) in enumerate(zip(all_ids, advantages)):
        completion_ids = seq_ids[0, prompt_len:]
        if len(completion_ids) == 0:
            continue

        # Build attention mask for the full generated sequence (no padding → all 1s)
        seq_mask = torch.ones_like(seq_ids)

        # 4. Current policy log probs over completion tokens
        logits = model(seq_ids, attention_mask=seq_mask).logits  # (1, seq, vocab)
        completion_logits = logits[0, prompt_len - 1:-1]         # align to completion tokens
        log_probs = F.log_softmax(completion_logits, dim=-1)
        token_log_probs = log_probs.gather(1, completion_ids.unsqueeze(-1)).squeeze(-1)

        # 5. Reference policy log probs (for KL)
        with torch.no_grad():
            ref_logits = ref_model(seq_ids, attention_mask=seq_mask).logits
            ref_completion_logits = ref_logits[0, prompt_len - 1:-1]
            ref_log_probs = F.log_softmax(ref_completion_logits, dim=-1)
            ref_token_log_probs = ref_log_probs.gather(1, completion_ids.unsqueeze(-1)).squeeze(-1)

        # 6. Ratio for PPO-style clipping (use sum of log probs → sequence ratio)
        old_log_prob = token_log_probs.detach().sum()
        new_log_prob = token_log_probs.sum()
        ratio = torch.exp(new_log_prob - old_log_prob)

        # 7. Clipped surrogate objective
        clipped_ratio = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS)
        policy_loss = -torch.min(ratio * advantage, clipped_ratio * advantage)

        # 8. KL penalty: KL(policy || ref) ≈ log_prob - ref_log_prob per token
        kl = (token_log_probs - ref_token_log_probs).mean()

        loss = policy_loss + KL_COEF * kl
        total_loss = total_loss + loss / G

    return total_loss, rewards, completions

In [ ]:
eval_items[0]["messages"][0]["content"]

In [ ]:
# Sanity check: verify the model responds as a chat model
test_messages = [
    {"role": "system", "content": "You are a helpful assistant. Answer the user's, thinking step by step."},
    {"role": "user", "content": eval_items[0]["messages"][1]["content"]},
]
input_ids, mask = apply_chat(test_messages)
prompt_len = input_ids.shape[1]
gold = eval_items[0]["answer"]

with torch.inference_mode():
    for j in range(4):
        out = model.generate(
            input_ids=input_ids,
            attention_mask=mask,
            max_new_tokens=2048,
            do_sample=True,
        )
        response = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)
        r = reward_fn(response, gold)
        print(f"── Sample {j+1} (reward={r:.1f}) ──")
        print(response)
        print()

print("="*80)
print("Gold answer:", gold)

In [ ]:
pre_acc = evaluate(eval_items, tag="pre-train eval")
print(f"Starting GRPO training... (pre-train eval acc: {pre_acc:.2%})")

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

# Pre-training eval
pre_acc = evaluate(eval_items, tag="pre-train eval")

# for epoch in range(EPOCHS):
#     print(f"\n{'='*60}\nEpoch {epoch + 1}/{EPOCHS}\n{'='*60}")
#     epoch_loss = 0.0

#     for step, item in enumerate(train_items):
#         optimizer.zero_grad()

#         print(f"\nStep {step + 1}/{len(train_items)} | gold={item['answer']!r}")
#         loss, rewards, completions = grpo_step(item["messages"], item["answer"])
#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#         optimizer.step()

#         epoch_loss += loss.item()
#         print(f"  rewards: {[round(float(r), 2) for r in rewards.tolist()]}")
#         print(f"  loss:    {loss.item():.4f}")

#     print(f"\nEpoch avg loss: {epoch_loss / len(train_items):.4f}")

# # Post-training eval
# post_acc = evaluate(eval_items, tag="post-train eval")

# print(f"\n{'='*60}")
# print(f"Accuracy: {pre_acc:.2%} → {post_acc:.2%} ({post_acc - pre_acc:+.2%})")
# print(f"{'='*60}")

# print("\nDone! Saving LoRA adapter...")
# model.save_pretrained("./grpo-lora")
# tokenizer.save_pretrained("./grpo-lora")